# MAP5935 - Statistical Learning (Chapter 11 - Survival Analysis)

**Prof. Christian Jäkel**

https://www.statlearning.com/

In [1]:
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots

from ISLP import load_data
from ISLP.models import ModelSpec as MS

from sklearn.linear_model import \
    (LinearRegression ,
    LogisticRegression ,
    Lasso)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline

from sklearn.model_selection import \
    (train_test_split ,
    GridSearchCV)

import torch
from torch import nn
from torch.optim import RMSprop
from torch.utils.data import TensorDataset

## *Conceptual Exercises*

### (8) Recall that the survival function $S(t)$, the hazard function $h(t)$, and the density function $f(t)$ are defined in equations $(1-3)$ below, respectively. Furthermore, define $F(t) = 1 − S(t)$.

###  Show that the following relationships hold:

$$ 
f(t)=dF(t)/dt
$$

$$ 
S(t)=\exp\Bigg(-\int_0^t h(u)du\Bigg)
$$

### Consider the following given relationships:
$$ 
S(t)=\Pr(T>t)\tag{1}
$$

$$
h(t)=\lim_{\Delta t\to 0}\frac{\Pr(t<T\leq t+\Delta t | T > t)}{\Delta t}\tag{2}
$$

$$
f(t)=\lim_{\Delta t\to 0}\frac{\Pr(t<T\leq t+\Delta t)}{\Delta t}\tag{3}
$$


**Solution** 

#### (1) Showing $ f(t) = \dfrac{dF(t)}{dt} $

For any $\Delta t>0$,

$$
F(t)=1-S(t)=1-\Pr(T>t)=P(T\leq t)
$$

Then we have

$$
F(t+\Delta t)-F(t)
= \Pr(T\le t+\Delta t) - \Pr(T\le t)
= \Pr(t< T \le t+\Delta t).
$$

Therefore, using the definition of $f(t)$ in equation $(3)$, we have:

$$
f(t)
= \lim_{\Delta t\to 0} \frac{\Pr(t< T\le t+\Delta t)}{\Delta t}
= \lim_{\Delta t\to 0} \frac{F(t+\Delta t)-F(t)}{\Delta t}
= \frac{dF(t)}{dt},
$$

i.e. $f$ is the derivative of the cdf $F$.

#### 2. Expressing $S(t)$ in terms of the hazard $h(t)$

First note that, using the definition of conditional probability,

$$
\Pr(t< T\le t+\Delta t \mid T>t)
= \frac{\Pr(t< T\le t+\Delta t ,\, T>t)}{\Pr(T>t)}
= \frac{\Pr(t< T\le t+\Delta t)}{S(t)}.
$$

Plugging this into the definition of the hazard,

$$
h(t)
= \lim_{\Delta t\to 0}
  \frac{ \Pr(t< T\le t+\Delta t \mid T>t)}{\Delta t}
= \lim_{\Delta t\to 0}
  \frac{ \Pr(t< T\le t+\Delta t)}{\Delta t\,S(t)}
= \frac{f(t)}{S(t)}.
$$

Thus

$$
f(t) = h(t)\,S(t).
$$

We also have $F(t)=1-S(t)$, so

$$
\frac{dS(t)}{dt}
= -\frac{dF(t)}{dt}
= -f(t)
= -h(t)\,S(t).
$$

Hence $S$ satisfies the differential equation

$$
\frac{dS(t)}{dt} = -h(t)\,S(t), \qquad S(0)=\Pr(T>0)=1.
$$

Solving it by separation of variables:

$$
\frac{dS(t)}{S(t)} = -h(t)\,dt
\quad\Longrightarrow\quad
\int_{S(0)}^{S(t)} \frac{1}{s}\,ds = -\int_0^t h(u)\,du,
$$

so

$$
\log S(t) - \log S(0) = -\int_0^t h(u)\,du.
$$

Since $S(0)=1$, $\log S(0)=0$, and we obtain

$$
\log S(t) = -\int_0^t h(u)\,du
\quad\Longrightarrow\quad
S(t) = \exp\!\left(-\int_0^t h(u)\,du\right).
$$

This proves the desired relationships.

## *Applied Exercises*

### (10) This exercise focuses on the brain tumor data, which is included in the `ISLP` library.

### (a) Plot the Kaplan-Meier survival curve with ±1 standard error bands, using the `KaplanMeierFitter()` estimator in the `lifelines` package.

**Solution** - According with the LAB $10.9.6$, we need do adapt it for a lag-$12$ representing the month for an RNN.

The following code was provided in the LAB.


In [2]:
NYSE = load_data('NYSE')
NYSE.columns

Index(['day_of_week', 'DJ_return', 'log_volume', 'log_volatility', 'train'], dtype='object')